In [1]:
import numpy as np
from pathlib import Path


# =============================================================================
# Path to the file saved by the previous script
# =============================================================================

RESULT_PATH = Path(
    "/home/maria/Science/results/human_adam_geometry_test/"
    "adam_human_loo_with_fold_directions.npz"
)

EPS = 1e-12


def normalize_rows(W: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(W, axis=1, keepdims=True)

    if np.any(norms < EPS):
        bad = np.where(norms.ravel() < EPS)[0]
        raise ValueError(f"Found near-zero decoder directions at folds: {bad}")

    return W / norms


def pairwise_angle_matrix_degrees(U: np.ndarray) -> np.ndarray:
    """
    U: shape (n_directions, n_features), row-normalized.

    Returns pairwise angles in degrees.
    """
    cos = U @ U.T
    cos = np.clip(cos, -1.0, 1.0)

    angles_rad = np.arccos(cos)
    angles_deg = np.degrees(angles_rad)

    return angles_deg


def main():
    if not RESULT_PATH.exists():
        raise FileNotFoundError(f"Could not find result file: {RESULT_PATH}")

    data = np.load(RESULT_PATH, allow_pickle=True)

    if "fold_weights_standardized" not in data.files:
        print("Available keys:")
        for k in data.files:
            print(" ", k)
        raise KeyError("Expected key 'fold_weights_standardized' not found.")

    W = np.asarray(data["fold_weights_standardized"], dtype=np.float64)

    if W.ndim != 2:
        raise ValueError(f"Expected fold_weights_standardized to be 2D, got {W.shape}")

    n_folds, n_features = W.shape

    print()
    print("=" * 80)
    print("LOO Adam decoder direction angle inspection")
    print("=" * 80)
    print(f"Loaded: {RESULT_PATH}")
    print(f"Fold directions shape: {W.shape}")
    print(f"Number of LOO directions: {n_folds}")
    print(f"Number of features: {n_features}")

    U = normalize_rows(W)
    angles = pairwise_angle_matrix_degrees(U)

    # Ignore diagonal self-angles.
    mask = ~np.eye(n_folds, dtype=bool)
    offdiag_angles = angles[mask]

    min_angle = float(offdiag_angles.min())
    max_angle = float(offdiag_angles.max())
    mean_angle = float(offdiag_angles.mean())
    median_angle = float(np.median(offdiag_angles))
    std_angle = float(offdiag_angles.std(ddof=1))

    min_pair = np.argwhere((angles == min_angle) & mask)[0]
    max_pair = np.argwhere((angles == max_angle) & mask)[0]

    # Cosine summaries too.
    cos = U @ U.T
    offdiag_cos = cos[mask]

    print()
    print("=" * 80)
    print("Pairwise angle summary, degrees")
    print("=" * 80)
    print(f"Min angle:    {min_angle:.6f} degrees, folds {tuple(min_pair)}")
    print(f"Max angle:    {max_angle:.6f} degrees, folds {tuple(max_pair)}")
    print(f"Mean angle:   {mean_angle:.6f} degrees")
    print(f"Median angle: {median_angle:.6f} degrees")
    print(f"Std angle:    {std_angle:.6f} degrees")

    print()
    print("=" * 80)
    print("Pairwise cosine summary")
    print("=" * 80)
    print(f"Max cosine:    {offdiag_cos.max():+.6f}")
    print(f"Min cosine:    {offdiag_cos.min():+.6f}")
    print(f"Mean cosine:   {offdiag_cos.mean():+.6f}")
    print(f"Median cosine: {np.median(offdiag_cos):+.6f}")
    print(f"Std cosine:    {offdiag_cos.std(ddof=1):.6f}")

    # Optional: also report orientation-invariant angles.
    # This treats u and -u as the same axis.
    abs_cos = np.abs(cos)
    abs_cos = np.clip(abs_cos, -1.0, 1.0)
    axis_angles = np.degrees(np.arccos(abs_cos))
    offdiag_axis_angles = axis_angles[mask]

    min_axis_angle = float(offdiag_axis_angles.min())
    max_axis_angle = float(offdiag_axis_angles.max())
    mean_axis_angle = float(offdiag_axis_angles.mean())
    median_axis_angle = float(np.median(offdiag_axis_angles))

    min_axis_pair = np.argwhere((axis_angles == min_axis_angle) & mask)[0]
    max_axis_pair = np.argwhere((axis_angles == max_axis_angle) & mask)[0]

    print()
    print("=" * 80)
    print("Orientation-invariant axis angle summary")
    print("=" * 80)
    print("This treats u and -u as the same axis.")
    print(f"Min axis angle:    {min_axis_angle:.6f} degrees, folds {tuple(min_axis_pair)}")
    print(f"Max axis angle:    {max_axis_angle:.6f} degrees, folds {tuple(max_axis_pair)}")
    print(f"Mean axis angle:   {mean_axis_angle:.6f} degrees")
    print(f"Median axis angle: {median_axis_angle:.6f} degrees")

    # Save full matrices for later plotting.
    out_path = RESULT_PATH.parent / "loo_decoder_pairwise_angles.npz"

    np.savez_compressed(
        out_path,
        angles_degrees=angles,
        cosines=cos,
        axis_angles_degrees=axis_angles,
        min_angle=min_angle,
        max_angle=max_angle,
        mean_angle=mean_angle,
        median_angle=median_angle,
        std_angle=std_angle,
        min_pair=min_pair,
        max_pair=max_pair,
        min_axis_angle=min_axis_angle,
        max_axis_angle=max_axis_angle,
        mean_axis_angle=mean_axis_angle,
        median_axis_angle=median_axis_angle,
        min_axis_pair=min_axis_pair,
        max_axis_pair=max_axis_pair,
    )

    print()
    print("=" * 80)
    print("Saved pairwise angle matrices")
    print("=" * 80)
    print(out_path)


if __name__ == "__main__":
    main()


LOO Adam decoder direction angle inspection
Loaded: /home/maria/Science/results/human_adam_geometry_test/adam_human_loo_with_fold_directions.npz
Fold directions shape: (118, 39209)
Number of LOO directions: 118
Number of features: 39209

Pairwise angle summary, degrees
Min angle:    34.801189 degrees, folds (np.int64(87), np.int64(109))
Max angle:    38.776343 degrees, folds (np.int64(14), np.int64(93))
Mean angle:   36.678939 degrees
Median angle: 36.648414 degrees
Std angle:    0.544585 degrees

Pairwise cosine summary
Max cosine:    +0.821137
Min cosine:    +0.779597
Mean cosine:   +0.801959
Median cosine: +0.802313
Std cosine:    0.005689

Orientation-invariant axis angle summary
This treats u and -u as the same axis.
Min axis angle:    34.801189 degrees, folds (np.int64(87), np.int64(109))
Max axis angle:    38.776343 degrees, folds (np.int64(14), np.int64(93))
Mean axis angle:   36.678939 degrees
Median axis angle: 36.648414 degrees

Saved pairwise angle matrices
/home/maria/Sci